# Sprint 5 · Webinar 20 · Sesión Práctica — Student Version  
## Caso: licores latinoamericanos

**Duración:** 100 minutos  
**Nivel:** principiante  
**Formato:** práctica guiada con ejercicios, notas y reflexión

Usa este notebook como **cuaderno de trabajo**. La meta no es copiar una solución final, sino:

- tomar apuntes durante la clase,
- completar código paso a paso,
- registrar resultados,
- comparar alternativas,
- escribir conclusiones cortas después de cada análisis.


## Ruta de la clase

### Estructura de la sesión

- **Ejercicio 0:** exploración inicial del dataset
- **Sección 1:** tratamiento de nulos
- **Sección 2:** detección de outliers con IQR
- **Sección 3:** análisis con `groupby()`
- **Sección 4:** visualizaciones rápidas con `.plot()`
- **Cierre:** take aways y reflexión final

### Datos de la sesión

Completa esta información antes o durante la clase:

- **Fecha:** `____ / ____ / 2025`
- **Estudiante:** `____________________________`
- **Instructor/a:** `____________________________`
- **Grupo / Cohorte:** `____________________________`


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---------------------------------------------------------
# Preparación de un dataset sintético para la práctica
# ---------------------------------------------------------
# Si el archivo no existe, lo creamos. Así el notebook
# puede ejecutarse de forma autónoma en cualquier equipo.
# ---------------------------------------------------------

ruta = Path("licores_latam_practica.csv")

if not ruta.exists():
    rng = np.random.default_rng(42)
    n = 40

    df_base = pd.DataFrame({
        "product_id": range(1, n + 1),
        "country": rng.choice(["Colombia", "México", "Perú", "Chile", "Argentina"], size=n),
        "category": rng.choice(["Ron", "Pisco", "Tequila", "Vino", "Cerveza"], size=n),
        "brand": [f"Marca_{i+1:02d}" for i in range(n)],
        "abv_percent": rng.integers(4, 45, size=n).astype(float),
        "bottle_volume_ml": rng.choice([330, 355, 500, 700, 750, 1000], size=n).astype(float),
        "price_usd": np.clip(rng.normal(18, 7, size=n).round(2), 4, None),
        "rating": np.clip(rng.normal(4.1, 0.4, size=n).round(1), 2.8, 4.9),
        "export_volume_liters": rng.integers(800, 20000, size=n).astype(float)
    })

    # Insertamos algunos valores faltantes para practicar limpieza
    df_base.loc[[2, 5], "price_usd"] = np.nan
    df_base.loc[[9, 15], "rating"] = np.nan
    df_base.loc[22, "country"] = np.nan
    df_base.loc[30, "category"] = np.nan
    df_base.loc[7, "abv_percent"] = np.nan

    # Insertamos dos outliers en el precio
    df_base.loc[1, "price_usd"] = 95
    df_base.loc[27, "price_usd"] = 120

    df_base.to_csv(ruta, index=False)

print(f"Archivo listo para la clase: {ruta.resolve()}")


## Ejercicio 0 · Exploración inicial del dataset

**Objetivo:** familiarizarnos con el caso de estudio antes de limpiar o analizar.

### Antes de ejecutar

Completa esta mini planificación:

| Pregunta | Mi respuesta |
|---|---|
| ¿Qué crees que representa cada fila? | |
| ¿Qué columnas parecen numéricas? | |
| ¿Qué columnas parecen categóricas o de texto? | |
| ¿Qué variable usarías primero para buscar outliers? | |
| ¿Qué columna podría tener nulos importantes para el negocio? | |

### Tu tarea

1. Carga el archivo usando `pd.read_csv()`.
2. Guarda el resultado en una variable llamada `df`.
3. Muestra las primeras filas con `head()`.


In [ ]:

# Ejercicio 0 — carga y vista inicial

# TODO 1: carga el archivo CSV
# df = pd.read_csv(ruta)

# TODO 2: muestra las primeras filas
# df.head()


### Preguntas guiadas

Después de explorar el dataset, responde:

- ¿Qué columnas parecen numéricas?
- ¿Qué columnas describen categorías o texto?
- ¿Qué variable te parece más útil para detectar outliers?
- ¿Qué columnas crees que podrían tener nulos problemáticos?
- ¿Qué primer análisis te gustaría hacer con este dataset?

### Evidencia rápida

| Método | ¿Para qué lo usé? | Hallazgo |
|---|---|---|
| `df.shape` | | |
| `df.info()` | | |
| `df.head()` | | |


# Sección 1 · Tratamiento de nulos

Los **valores nulos** o faltantes aparecen cuando una observación no tiene dato registrado.

En esta parte vas a comparar varias estrategias:

- detectar nulos,
- inspeccionar filas incompletas,
- eliminar registros,
- imputar valores,
- decidir una estrategia final de trabajo.

> Mientras avanzas, toma nota de **qué cambia en el dataset** y **por qué una estrategia puede ser mejor que otra**.


## Ejercicio 1 · Contar valores faltantes

**Objetivo:** identificar cuántos nulos hay por columna.

### Pista
Usa `isna().sum()` sobre el DataFrame.

### Registro
Antes de ejecutar, escribe tu predicción:

| Columna | ¿Espero nulos? | ¿Por qué? |
|---|---|---|
| `country` | | |
| `category` | | |
| `price_usd` | | |
| `rating` | | |
| `abv_percent` | | |


In [ ]:

# Ejercicio 1 — conteo de nulos

# TODO:
# calcula cuántos valores faltantes hay por columna
# Ejemplo esperado:
# df.isna().sum()


## Ejercicio 2 · Ver las filas que tienen al menos un nulo

**Objetivo:** inspeccionar qué registros están incompletos.

### Tu tarea

1. Filtra las filas que tengan al menos un valor nulo.
2. Guarda el resultado en una variable llamada `filas_con_nulos`.
3. Muestra la tabla resultante.

### Preguntas de apoyo

- ¿Hay muchas filas afectadas o pocas?
- ¿Los nulos se concentran en una sola columna o en varias?
- ¿Eliminarías estas filas de inmediato?


In [ ]:

# Ejercicio 2 — inspección de filas con nulos

# TODO:
# crea una tabla con las filas que tengan al menos un nulo
# guarda el resultado en una variable llamada filas_con_nulos

# filas_con_nulos = ...
# filas_con_nulos


## Ejercicio 3 · Primer acercamiento con `dropna()`

**Objetivo:** entender qué pasa si eliminamos todas las filas con algún nulo.

### Tu tarea

1. Crea una copia del DataFrame sin filas con nulos.
2. Guarda el resultado en `df_drop_total`.
3. Compara el número de filas antes y después.

### Análisis esperado

- ¿Cuántas filas se eliminaron?
- ¿Te parece una pérdida pequeña o grande?
- ¿En un caso real aplicarías esta estrategia sin revisar más?


In [ ]:

# Ejercicio 3 — eliminación total de filas con nulos

# TODO:
# crea una versión del dataset sin filas con ningún nulo
# df_drop_total = ...

# TODO:
# imprime filas originales y filas después de dropna()
# print("Filas originales:", ...)
# print("Filas después de dropna():", ...)


## Ejercicio 4 · `dropna()` solo en columnas clave

**Objetivo:** aplicar una eliminación más controlada.

### Contexto

A veces una fila solo debe eliminarse si falta una columna crítica para el análisis.  
En este caso, piensa en columnas como `country` y `category`.

### Tu tarea

1. Aplica `dropna()` solo sobre columnas clave.
2. Guarda el resultado en `df_drop_subset`.
3. Compara el tamaño antes y después.

### Completa esta tabla

| Estrategia | Filas finales | ¿La usaría? | ¿Por qué? |
|---|---:|---|---|
| `dropna()` total | | | |
| `dropna(subset=...)` | | | |


In [ ]:

# Ejercicio 4 — eliminación condicional

# TODO:
# elimina filas solo si faltan columnas clave como country o category
# df_drop_subset = df.dropna(subset=[...])

# TODO:
# compara tamaños
# print("Filas originales:", ...)
# print("Filas después del filtro:", ...)


## Ejercicio 5 · Imputación numérica con media y mediana

**Objetivo:** comparar dos formas clásicas de imputar datos numéricos.

### Recordatorio teórico

- La **media** usa el promedio.
- La **mediana** usa el valor central.
- La mediana suele ser más robusta cuando hay outliers.

### Tu tarea

1. Crea dos copias del DataFrame: una para imputar con media y otra con mediana.
2. Calcula la media y la mediana de `price_usd`.
3. Rellena los nulos de `price_usd` en cada copia.
4. Muestra o imprime los valores usados.

### Predicción

Marca cuál crees que será más conveniente en este caso:

- [ ] Media
- [ ] Mediana

Explica en una frase por qué.


In [ ]:

# Ejercicio 5 — imputación con media y mediana

# TODO:
# crea dos copias del DataFrame
# df_media = ...
# df_mediana = ...

# TODO:
# calcula media y mediana de price_usd
# media_precio = ...
# mediana_precio = ...

# TODO:
# rellena los nulos de price_usd en cada copia
# df_media["price_usd"] = ...
# df_mediana["price_usd"] = ...

# TODO:
# imprime los valores usados
# print("Media:", ...)
# print("Mediana:", ...)


## Ejercicio 6 · Imputación final del dataset de trabajo

**Objetivo:** construir una versión limpia para seguir con la clase.

### Estrategia sugerida

- `country` y `category`: imputar con la moda.
- `price_usd`, `rating`, `abv_percent`: imputar con la mediana.

### Tu tarea

1. Crea una copia llamada `df_limpio`.
2. Calcula los valores de imputación.
3. Aplica `fillna()` en las columnas correspondientes.
4. Verifica al final si quedaron nulos.

### Checklist

- [ ] Creé una copia del dataset original.
- [ ] Imputé texto con moda.
- [ ] Imputé variables numéricas con mediana.
- [ ] Verifiqué que no quedaran nulos relevantes.


In [ ]:

# Ejercicio 6 — versión limpia para continuar

# TODO:
# crea copia de trabajo
# df_limpio = ...

# TODO:
# calcula moda para country y category
# moda_country = ...
# moda_category = ...

# TODO:
# calcula medianas para variables numéricas
# mediana_price = ...
# mediana_rating = ...
# mediana_abv = ...

# TODO:
# completa los nulos con fillna()

# TODO:
# valida que ya no haya nulos
# df_limpio.isna().sum()


### Mini-análisis de la Sección 1

Completa estas respuestas con base en tus resultados:

1. ¿Qué estrategia te pareció más agresiva: `dropna()` o imputar?
2. ¿En qué caso usarías mediana en lugar de media?
3. ¿Qué riesgo existe si imputamos sin entender el contexto del dato?
4. ¿Qué versión del dataset usarás para seguir y por qué?

| Decisión | Mi respuesta |
|---|---|
| Estrategia elegida | |
| Columna más delicada | |
| Riesgo principal | |


# Sección 2 · Outliers con IQR

Un **outlier** es un valor que se aleja mucho del comportamiento general de los datos.

En esta práctica usarás el método **IQR**:

- **Q1:** percentil 25
- **Q3:** percentil 75
- **IQR:** `Q3 - Q1`
- **Límite inferior:** `Q1 - 1.5 * IQR`
- **Límite superior:** `Q3 + 1.5 * IQR`

### Objetivo analítico

Detectar posibles valores extremos en `price_usd` y decidir si conviene excluirlos del análisis.


## Ejercicio 7 · Calcular Q1, Q3 e IQR en `price_usd`

**Objetivo:** construir la base matemática para detectar outliers.

### Pista
Usa `quantile(0.25)` y `quantile(0.75)`.

### Registro del estudiante

| Métrica | Resultado |
|---|---|
| Q1 | |
| Q3 | |
| IQR | |
| Límite inferior | |
| Límite superior | |


In [ ]:

# Ejercicio 7 — cálculo del IQR

# TODO:
# calcula Q1 y Q3 sobre price_usd en df_limpio
# Q1 = ...
# Q3 = ...

# TODO:
# calcula IQR y límites
# IQR = ...
# limite_inferior = ...
# limite_superior = ...

# TODO:
# imprime o muestra los resultados
# print("Q1:", Q1)
# print("Q3:", Q3)
# print("IQR:", IQR)
# print("Límite inferior:", limite_inferior)
# print("Límite superior:", limite_superior)


## Ejercicio 8 · Identificar los outliers

**Objetivo:** ver cuáles productos tienen un precio fuera del rango esperado.

### Tu tarea

1. Crea una condición booleana para detectar outliers en `price_usd`.
2. Guarda la condición en `es_outlier`.
3. Crea un DataFrame llamado `outliers_precio`.
4. Muestra columnas útiles para revisar esos registros.

### Qué observar

- ¿Cuántos outliers aparecen?
- ¿Son productos realmente extraños o podrían ser productos premium?
- ¿Qué columnas ayudarían a justificar una decisión de negocio?


In [ ]:

# Ejercicio 8 — detección de outliers

# TODO:
# crea la condición booleana
# es_outlier = ...

# TODO:
# filtra los outliers y guárdalos en outliers_precio
# outliers_precio = ...

# TODO:
# muestra una selección útil de columnas
# outliers_precio[[...]]


## Ejercicio 9 · Crear una versión sin outliers

**Objetivo:** preparar el dataset para análisis más estables.

### Tu tarea

1. Usa la condición `es_outlier` para excluir esos registros.
2. Guarda el resultado en `df_sin_outliers`.
3. Imprime cuántas filas eliminaste y el tamaño final del dataset.

### Reflexión

Completa esta tabla:

| Pregunta | Mi respuesta |
|---|---|
| ¿Cuántos outliers eliminé? | |
| ¿Eliminar outliers cambia mucho el tamaño del dataset? | |
| ¿Usaría esta decisión en un entorno real sin más validaciones? | |


In [ ]:

# Ejercicio 9 — dataset sin outliers

# TODO:
# crea una versión del dataset excluyendo outliers
# df_sin_outliers = ...

# TODO:
# imprime cuántas filas fueron eliminadas y tamaño final
# print("Filas con outliers eliminadas:", ...)
# print("Tamaño final del dataset:", ...)


### Mini-análisis de la Sección 2

Responde con base en tu análisis:

- ¿Cuántos outliers aparecieron?
- ¿Crees que esos valores son errores o productos premium?
- ¿Qué podría pasar con el promedio del precio si dejas esos extremos?
- ¿Qué harías si el profesor o el negocio te pidiera justificar la decisión?

| Decisión sobre outliers | Justificación |
|---|---|
| Los eliminaría | |
| Los dejaría | |
| Haría análisis en paralelo | |


# Sección 3 · GroupBy

`groupby()` sirve para **agrupar datos** y luego resumirlos.

En esta parte vas a practicar tres tipos de resumen:

1. conteos,
2. promedios,
3. comparación de varias métricas numéricas.

> Trabaja sobre `df_sin_outliers` para que los resultados sean más estables.


## Ejercicio 10 · Contar productos por país

**Objetivo:** hacer un primer `groupby()` simple.

### Tu tarea

1. Agrupa por `country`.
2. Cuenta cuántos `product_id` hay por país.
3. Ordena de mayor a menor.
4. Guarda el resultado en `productos_por_pais`.

### Antes de ejecutar

¿Cuál país crees que tendrá más registros? Escribe tu hipótesis:
`________________________________________________________`


In [ ]:

# Ejercicio 10 — conteo por país

# TODO:
# agrupa por country y cuenta product_id
# productos_por_pais = ...

# TODO:
# ordénalo de mayor a menor y muéstralo
# productos_por_pais


## Ejercicio 11 · Precio promedio por categoría

**Objetivo:** usar `groupby()` con una métrica de promedio.

### Pregunta de negocio

¿Qué categoría tiene, en promedio, el precio más alto?

### Tu tarea

1. Agrupa por `category`.
2. Calcula el promedio de `price_usd`.
3. Redondea a 2 decimales.
4. Ordena de mayor a menor.

### Registro

| Categoría más cara | Precio promedio | ¿Me sorprende? |
|---|---:|---|
| | | |


In [ ]:

# Ejercicio 11 — precio promedio por categoría

# TODO:
# calcula el precio promedio por categoría
# precio_promedio_categoria = ...

# TODO:
# redondea, ordena y muestra
# precio_promedio_categoria


## Ejercicio 12 · Dos variables numéricas resumidas por país

**Objetivo:** comparar el precio promedio y el ABV promedio en una sola tabla.

### Tu tarea

1. Agrupa por `country`.
2. Selecciona `price_usd` y `abv_percent`.
3. Calcula el promedio.
4. Redondea a 2 decimales.

### Preguntas

- ¿Ves un país con mayor precio promedio?
- ¿Ese mismo país también tiene mayor `abv_percent` promedio?
- ¿Crees que existe una relación clara entre ambas métricas?


In [ ]:

# Ejercicio 12 — resumen de dos métricas por país

# TODO:
# construye una tabla resumen por país
# resumen_pais = ...

# TODO:
# muestra la tabla
# resumen_pais


### Mini-análisis de la Sección 3

Completa las respuestas después de revisar tus tablas:

1. ¿Qué país tiene más productos en el dataset?
2. ¿Qué categoría muestra mayor precio promedio?
3. ¿Observas alguna relación aparente entre precio promedio y ABV promedio?
4. ¿Qué tabla le mostrarías primero a un stakeholder no técnico?

| Tabla | ¿Para qué sirve? | ¿La mostraría en un reporte? |
|---|---|---|
| `productos_por_pais` | | |
| `precio_promedio_categoria` | | |
| `resumen_pais` | | |


# Sección 4 · Gráficas con `.plot()`

En `pandas`, `.plot()` permite crear visualizaciones rápidas sin escribir mucho código.

En esta sección vas a construir cuatro gráficos:

- barras,
- histograma,
- boxplot,
- dispersión.

### Recomendación

Antes de ejecutar cada gráfico, intenta responder:  
**¿qué historia espero ver en la visualización?**


## Ejercicio 13 · Gráfica de barras

**Objetivo:** visualizar el precio promedio por país.

### Pregunta analítica

¿Cuál es el país con mayor precio promedio?

### Tu tarea

1. Crea la serie `precio_promedio_pais`.
2. Ordénala.
3. Constrúyela como gráfico de barras.
4. Agrega título y etiqueta del eje `y`.

### Observaciones del estudiante

- País con mayor precio promedio: `______________________`
- ¿La diferencia entre países parece grande o pequeña? `______________________`


In [ ]:

# Ejercicio 13 — gráfico de barras

# TODO:
# calcula precio promedio por país
# precio_promedio_pais = ...

# TODO:
# crea el gráfico de barras
# ax = precio_promedio_pais.plot(...)

# TODO:
# agrega etiquetas y muestra el gráfico
# ax.set_ylabel("...")
# plt.tight_layout()
# plt.show()


## Ejercicio 14 · Histograma

**Objetivo:** observar cómo se distribuye `price_usd`.

### Preguntas antes de ejecutar

- ¿Crees que la distribución estará concentrada en pocos rangos?
- ¿Esperas una cola hacia valores altos?
- ¿Eliminar outliers debería cambiar la forma del histograma?

### Tu tarea

Construye un histograma de `price_usd` usando `df_sin_outliers`.


In [ ]:

# Ejercicio 14 — histograma

# TODO:
# crea un histograma de price_usd
# ax = df_sin_outliers["price_usd"].plot(...)

# TODO:
# etiqueta el eje x y muestra el gráfico
# ax.set_xlabel("...")
# plt.tight_layout()
# plt.show()


## Ejercicio 15 · Boxplot

**Objetivo:** revisar la dispersión del precio y detectar posibles extremos.

### Tu tarea

1. Construye un boxplot para `price_usd`.
2. Agrega un título.
3. Etiqueta el eje `y`.

### Interpreta

- ¿Dónde parece estar la mediana?
- ¿La caja se ve ancha o estrecha?
- ¿Aun después de limpiar quedan valores alejados?


In [ ]:

# Ejercicio 15 — boxplot

# TODO:
# crea un boxplot para price_usd
# ax = df_sin_outliers[[...]].plot(...)

# TODO:
# agrega etiqueta y muestra el gráfico
# ax.set_ylabel("...")
# plt.tight_layout()
# plt.show()


## Ejercicio 16 · Gráfico de dispersión

**Objetivo:** comparar `abv_percent` con `price_usd`.

### Pregunta analítica

¿Los productos con más graduación alcohólica tienden a ser más costosos?

### Tu tarea

1. Crea un scatter plot usando `abv_percent` en `x` y `price_usd` en `y`.
2. Agrega título y etiquetas.
3. Observa si hay una tendencia clara.

### Registro

| Observación | Mi respuesta |
|---|---|
| ¿Veo relación positiva, negativa o ninguna? | |
| ¿Hay mucha dispersión? | |
| ¿Mostraría este gráfico a negocio? | |


In [ ]:

# Ejercicio 16 — dispersión entre ABV y precio

# TODO:
# crea el scatter plot
# ax = df_sin_outliers.plot(
#     kind="scatter",
#     x="abv_percent",
#     y="price_usd",
#     figsize=(6, 4),
#     title="..."
# )

# TODO:
# agrega etiquetas y muestra el gráfico
# ax.set_xlabel("...")
# ax.set_ylabel("...")
# plt.tight_layout()
# plt.show()


## Take aways

Completa con tus propias palabras:

1. **Nulos:**  
   `____________________________________________________________________`

2. **Outliers:**  
   `____________________________________________________________________`

3. **`groupby()`:**  
   `____________________________________________________________________`

4. **Visualización rápida con pandas:**  
   `____________________________________________________________________`

### Checklist final de aprendizaje

| Habilidad | ¿La practiqué? | Evidencia |
|---|---|---|
| Contar y revisar nulos | ☐ | |
| Imputar datos faltantes | ☐ | |
| Detectar outliers con IQR | ☐ | |
| Resumir información con `groupby()` | ☐ | |
| Crear gráficos con `.plot()` | ☐ | |


## Cierre

### Lo que logramos hoy

- Exploramos un dataset realista y pequeño.
- Detectamos y tratamos valores faltantes.
- Identificamos outliers con IQR.
- Resumimos información con `groupby()`.
- Construimos visualizaciones iniciales para análisis exploratorio.

### Reflexión final

Responde en una frase o dos:

1. ¿Qué parte te resultó más clara?
2. ¿Qué parte te costó más?
3. ¿Qué paso repetirías para reforzar tu aprendizaje?
4. Si tuvieras que explicar este notebook a otra persona, ¿qué bloque explicarías primero?

### Próximo paso sugerido

En una celda nueva, intenta crear un análisis adicional por tu cuenta.  
Por ejemplo:

- precio promedio por `country` y `category`,
- rating promedio por categoría,
- total exportado por país,
- o una nueva gráfica simple.

### ¿Necesitas ayuda?
- `DATA-CO-LEARNING`: revisa los horarios de apoyo en la guía del estudiante.
- `DA_CONSULTA`: publica dudas de contenido o del proyecto usando el tag correspondiente.
- `SPRINT FOCUS`: sesiones extra para profundizar temas clave del sprint.
- `SESIONES 1:1`: agenda con anticipación si necesitas apoyo individual.

In [ ]:

# Espacio libre para práctica adicional

# Usa esta celda para crear un análisis extra por tu cuenta.
# Ideas:
# 1) rating promedio por categoría
# 2) export_volume_liters promedio por país
# 3) tabla combinada country + category
# 4) una gráfica adicional
